# Importing Libraries

In [ ]:
from io import BytesIO
import os
import zipfile
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Reshape, Bidirectional, LSTM, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from albumentations import Compose, Rotate, GaussNoise, MotionBlur

# Dataset Download


In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d robikscube/textocr-text-extraction-from-images-dataset

# Extracting Dataset

In [ ]:
with zipfile.ZipFile('/content/textocr-text-extraction-from-images-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

In [ ]:
annotations_file = '/content/dataset/TextOCR_0.1_train.json'
images_dir = '/content/dataset/train_val_images/train_images'

In [ ]:
with open(annotations_file, 'r') as f:
    annotations = json.load(f)

In [ ]:
for k,v in annotations['imgs'].items():
    print(v)

In [ ]:
for k,v in annotations['imgs'].items():
    image_path = os.path.join(images_dir, v['file_name'])
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image_ann = [a for a in v['file_name'] if ]

In [ ]:
images = []
labels = []

for k,v in annotations['imgs'].items():
    image_path = os.path.join(images_dir, v['file_name'])
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    image_annotations = [a for a in annotations['file_name'] if a['image_id'] == item['id']]

    for annotation in image_annotations:
        bbox = annotation['bbox']
        text = annotation['text']

        x, y, w, h = map(int, bbox)
        roi = image[y:y+h, x:x+w]

        images.append(roi)
        labels.append(text)

In [ ]:
def preprocess_image(image):
    image_blurred = cv2.GaussianBlur(image, (5, 5), 0)
    edges = cv2.Canny(image_blurred, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for contour in contours:
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
            rect = cv2.minAreaRect(pts)
            box = cv2.boxPoints(rect)
            box = np.int0(box)

            box = sorted(box, key=lambda x: (x[1], x[0]))

            width = int(max(rect[1][0], rect[1][1]))
            height = int(min(rect[1][0], rect[1][1]))

            top_left, top_right, bottom_right, bottom_left = box

            if width < height:
                top_left, top_right, bottom_right, bottom_left = top_right, bottom_right, bottom_left, top_left

            dst = np.array([
                [0, 0],
                [width - 1, 0],
                [width - 1, height - 1],
                [0, height - 1]], dtype="float32")

            M = cv2.getPerspectiveTransform(np.array([top_left, top_right, bottom_right, bottom_left], dtype="float32"), dst)
            warped = cv2.warpPerspective(image, M, (width, height))

            return warped

    return image

In [ ]:
def preprocess_images(images):
    preprocessed_images = []
    for img in images:
        preprocessed_image = preprocess_image(img)
        preprocessed_images.append(preprocessed_image)
    return preprocessed_images

In [ ]:
preprocessed_images = preprocess_images(images)

In [ ]:
def augment_image(image):
    # Define augmentation pipeline
    transform = Compose([
        Rotate(limit=10, p=0.5),
        GaussNoise(var_limit=(10.0, 50.0), p=0.5),
        MotionBlur(blur_limit=3, p=0.5),
    ])
    augmented = transform(image=image)['image']
    return augmented

def augment_images(images):
    return [augment_image(img) for img in images]

In [ ]:
augmented_images = augment_images(preprocessed_images)

In [ ]:
char_list = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
char_to_idx = {char: idx + 1 for idx, char in enumerate(char_list)}

In [ ]:
def encode_labels(labels):
    encoded_labels = []
    for label in labels:
        encoded = [char_to_idx[char] for char in label if char in char_to_idx]
        encoded_labels.append(encoded)
    return pad_sequences(encoded_labels, padding='post')

In [ ]:
encoded_labels = encode_labels(labels)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(augmented_images, encoded_labels, test_size=0.2, random_state=42)

In [ ]:
X_train_np = np.array(X_train) / 255.0
X_test_np = np.array(X_test) / 255.0
y_train_np = np.array(y_train)
y_test_np = np.array(y_test)

In [ ]:
X_train_np = X_train_np[..., np.newaxis]
X_test_np = X_test_np[..., np.newaxis]

In [ ]:
def build_ocr_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)

    # Feature extraction with CNN
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    # Reshape for RNN
    x = Reshape(target_shape=(-1, x.shape[-1]))(x)

    # Sequence prediction with Bidirectional RNN
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dense(64, activation='relu')(x)
    x = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, x)
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model

In [ ]:
num_classes = len(char_list) + 1  # +1 for the CTC blank token
input_shape = (None, None, 1)

In [ ]:
model = build_ocr_model(input_shape, num_classes)
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
history = model.fit(
    X_train_np, y_train_np,
    validation_data=(X_test_np, y_test_np),
    batch_size=32,
    epochs=50,
    callbacks=[early_stopping]
)

In [ ]:
plot_training(history)